In [0]:
-- MultiLoad log, work, error, and release commands were removed because they are Teradata utility artifacts.
-- The source database qualifier was removed because no target Unity Catalog mapping was provided.
DELETE FROM DLYTXN_DATA_RAW;

-- The source database qualifier was removed because no target Unity Catalog mapping was provided.
DELETE FROM DLYTXN_CTRL_RAW;

-- TODO: manual review required — confirm the cloud source path and implement the MultiLoad 500-reject limit plus MARK/IGNORE duplicate-row handling in the workflow.
-- MultiLoad was replaced with COPY INTO staging followed by insert-only MERGE statements.
CREATE OR REPLACE TEMP TABLE stg_dailytxn_raw (
    raw_line STRING
);

-- The fixed-width source is staged as one column by using a delimiter that does not occur in the extract.
COPY INTO stg_dailytxn_raw
FROM '../data/dailytxn.dat'
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'false',
    'delimiter' = '\u001F',
    'mode' = 'FAILFAST'
)
COPY_OPTIONS ('force' = 'true');

-- The MultiLoad detail layout was replaced with fixed-position SUBSTRING expressions.
CREATE OR REPLACE TEMP VIEW stg_dailytxn_detail AS
SELECT
    SUBSTRING(raw_line, 1, 1) AS RECTYPE,
    SUBSTRING(raw_line, 2, 4) AS SUPPLIER_CD,
    SUBSTRING(raw_line, 6, 9) AS FILE_SEQ_NR,
    SUBSTRING(raw_line, 15, 35) AS TXN_REF,
    SUBSTRING(raw_line, 50, 4) AS TXN_SEQ_NR,
    SUBSTRING(raw_line, 54, 6) AS SORT_CD,
    SUBSTRING(raw_line, 60, 8) AS ACCT_NR,
    SUBSTRING(raw_line, 68, 4) AS TXN_TYP_CD,
    SUBSTRING(raw_line, 72, 3) AS CHNL_CD,
    SUBSTRING(raw_line, 75, 1) AS DR_CR_IND,
    SUBSTRING(raw_line, 76, 16) AS TXN_AMT,
    SUBSTRING(raw_line, 92, 3) AS TXN_CCY,
    SUBSTRING(raw_line, 95, 16) AS ORIG_AMT,
    SUBSTRING(raw_line, 111, 3) AS ORIG_CCY,
    SUBSTRING(raw_line, 114, 8) AS BOOKING_DT,
    SUBSTRING(raw_line, 122, 8) AS VALUE_DT,
    SUBSTRING(raw_line, 130, 14) AS TXN_TMS,
    SUBSTRING(raw_line, 144, 140) AS NARRATIVE,
    SUBSTRING(raw_line, 284, 140) AS CPTY_NM,
    SUBSTRING(raw_line, 424, 6) AS CPTY_SORT_CD,
    SUBSTRING(raw_line, 430, 8) AS CPTY_ACCT_NR,
    SUBSTRING(raw_line, 438, 34) AS CPTY_IBAN,
    SUBSTRING(raw_line, 472, 11) AS CPTY_BIC,
    SUBSTRING(raw_line, 483, 2) AS CPTY_BANK_CTRY,
    SUBSTRING(raw_line, 485, 2) AS CPTY_CTRY,
    SUBSTRING(raw_line, 487, 140) AS ORDERING_CUST_NM,
    SUBSTRING(raw_line, 627, 140) AS BENEF_CUST_NM,
    SUBSTRING(raw_line, 767, 35) AS PAYMENT_REF,
    SUBSTRING(raw_line, 802, 35) AS E2E_ID,
    SUBSTRING(raw_line, 837, 36) AS UETR,
    SUBSTRING(raw_line, 873, 3) AS SWIFT_MSG_TYP,
    SUBSTRING(raw_line, 876, 3) AS CHARGE_BEARER,
    SUBSTRING(raw_line, 879, 11) AS INTERMED_BIC,
    SUBSTRING(raw_line, 890, 4) AS MCC,
    SUBSTRING(raw_line, 894, 40) AS MERCHANT_NM,
    SUBSTRING(raw_line, 934, 2) AS MERCHANT_CTRY,
    SUBSTRING(raw_line, 936, 15) AS MERCHANT_ID,
    SUBSTRING(raw_line, 951, 8) AS TERMINAL_ID,
    SUBSTRING(raw_line, 959, 8) AS ATM_ID,
    SUBSTRING(raw_line, 967, 19) AS CARD_TOKEN,
    SUBSTRING(raw_line, 986, 2) AS POS_ENTRY_MODE,
    SUBSTRING(raw_line, 988, 1) AS CARD_PRESENT_IND,
    SUBSTRING(raw_line, 989, 6) AS BRANCH_SORT_CD,
    SUBSTRING(raw_line, 995, 8) AS TELLER_ID,
    SUBSTRING(raw_line, 1003, 1) AS CASH_IND,
    SUBSTRING(raw_line, 1004, 6) AS CHEQUE_NR,
    SUBSTRING(raw_line, 1010, 40) AS DEVICE_ID,
    SUBSTRING(raw_line, 1050, 45) AS IP_ADDR,
    SUBSTRING(raw_line, 1095, 2) AS IP_CTRY,
    SUBSTRING(raw_line, 1097, 36) AS SESSION_ID,
    SUBSTRING(raw_line, 1133, 2) AS AUTH_METHOD,
    SUBSTRING(raw_line, 1135, 2) AS COP_RESULT,
    SUBSTRING(raw_line, 1137, 1) AS APP_FRAUD_IND,
    SUBSTRING(raw_line, 1138, 4) AS RETURN_REASON_CD,
    SUBSTRING(raw_line, 1142, 1) AS REVERSAL_IND,
    SUBSTRING(raw_line, 1143, 35) AS ORIG_TXN_REF,
    SUBSTRING(raw_line, 1178, 16) AS BAL_AFTER,
    SUBSTRING(raw_line, 1194, 2) AS TXN_STATUS,
    SUBSTRING(raw_line, 1196, 16) AS FEE_AMT,
    SUBSTRING(raw_line, 1212, 4) AS PURPOSE_CD,
    SUBSTRING(raw_line, 1216, 20) AS BATCH_ID,
    SUBSTRING(raw_line, 1236, 8) AS EXTRACT_DT
FROM stg_dailytxn_raw
WHERE SUBSTRING(raw_line, 1, 1) = 'D';

-- The MultiLoad header/trailer layout was replaced with fixed-position SUBSTRING expressions.
CREATE OR REPLACE TEMP VIEW stg_dailytxn_control AS
SELECT
    SUBSTRING(raw_line, 1, 1) AS RECTYPE,
    SUBSTRING(raw_line, 2, 4) AS SUPPLIER_CD,
    SUBSTRING(raw_line, 6, 20) AS FILE_ID,
    SUBSTRING(raw_line, 26, 8) AS EXTRACT_DT,
    SUBSTRING(raw_line, 34, 10) AS REC_CNT,
    SUBSTRING(raw_line, 44, 18) AS DR_TOTAL,
    SUBSTRING(raw_line, 62, 18) AS CR_TOTAL,
    SUBSTRING(raw_line, 80, 14) AS CREATED_TMS
FROM stg_dailytxn_raw
WHERE SUBSTRING(raw_line, 1, 1) IN ('H', 'T');

-- The detail MultiLoad INSERT was converted to an insert-only Delta MERGE after the target clear.
MERGE INTO DLYTXN_DATA_RAW AS t
USING stg_dailytxn_detail AS s
ON FALSE
WHEN NOT MATCHED THEN INSERT (
    RECTYPE,
    SUPPLIER_CD,
    FILE_SEQ_NR,
    TXN_REF,
    TXN_SEQ_NR,
    SORT_CD,
    ACCT_NR,
    TXN_TYP_CD,
    CHNL_CD,
    DR_CR_IND,
    TXN_AMT,
    TXN_CCY,
    ORIG_AMT,
    ORIG_CCY,
    BOOKING_DT,
    VALUE_DT,
    TXN_TMS,
    NARRATIVE,
    CPTY_NM,
    CPTY_SORT_CD,
    CPTY_ACCT_NR,
    CPTY_IBAN,
    CPTY_BIC,
    CPTY_BANK_CTRY,
    CPTY_CTRY,
    ORDERING_CUST_NM,
    BENEF_CUST_NM,
    PAYMENT_REF,
    E2E_ID,
    UETR,
    SWIFT_MSG_TYP,
    CHARGE_BEARER,
    INTERMED_BIC,
    MCC,
    MERCHANT_NM,
    MERCHANT_CTRY,
    MERCHANT_ID,
    TERMINAL_ID,
    ATM_ID,
    CARD_TOKEN,
    POS_ENTRY_MODE,
    CARD_PRESENT_IND,
    BRANCH_SORT_CD,
    TELLER_ID,
    CASH_IND,
    CHEQUE_NR,
    DEVICE_ID,
    IP_ADDR,
    IP_CTRY,
    SESSION_ID,
    AUTH_METHOD,
    COP_RESULT,
    APP_FRAUD_IND,
    RETURN_REASON_CD,
    REVERSAL_IND,
    ORIG_TXN_REF,
    BAL_AFTER,
    TXN_STATUS,
    FEE_AMT,
    PURPOSE_CD,
    BATCH_ID,
    EXTRACT_DT
)
VALUES (
    s.RECTYPE,
    s.SUPPLIER_CD,
    s.FILE_SEQ_NR,
    s.TXN_REF,
    s.TXN_SEQ_NR,
    s.SORT_CD,
    s.ACCT_NR,
    s.TXN_TYP_CD,
    s.CHNL_CD,
    s.DR_CR_IND,
    s.TXN_AMT,
    s.TXN_CCY,
    s.ORIG_AMT,
    s.ORIG_CCY,
    s.BOOKING_DT,
    s.VALUE_DT,
    s.TXN_TMS,
    s.NARRATIVE,
    s.CPTY_NM,
    s.CPTY_SORT_CD,
    s.CPTY_ACCT_NR,
    s.CPTY_IBAN,
    s.CPTY_BIC,
    s.CPTY_BANK_CTRY,
    s.CPTY_CTRY,
    s.ORDERING_CUST_NM,
    s.BENEF_CUST_NM,
    s.PAYMENT_REF,
    s.E2E_ID,
    s.UETR,
    s.SWIFT_MSG_TYP,
    s.CHARGE_BEARER,
    s.INTERMED_BIC,
    s.MCC,
    s.MERCHANT_NM,
    s.MERCHANT_CTRY,
    s.MERCHANT_ID,
    s.TERMINAL_ID,
    s.ATM_ID,
    s.CARD_TOKEN,
    s.POS_ENTRY_MODE,
    s.CARD_PRESENT_IND,
    s.BRANCH_SORT_CD,
    s.TELLER_ID,
    s.CASH_IND,
    s.CHEQUE_NR,
    s.DEVICE_ID,
    s.IP_ADDR,
    s.IP_CTRY,
    s.SESSION_ID,
    s.AUTH_METHOD,
    s.COP_RESULT,
    s.APP_FRAUD_IND,
    s.RETURN_REASON_CD,
    s.REVERSAL_IND,
    s.ORIG_TXN_REF,
    s.BAL_AFTER,
    s.TXN_STATUS,
    s.FEE_AMT,
    s.PURPOSE_CD,
    s.BATCH_ID,
    s.EXTRACT_DT
);

-- The control MultiLoad INSERT was converted to an insert-only Delta MERGE after the target clear.
MERGE INTO DLYTXN_CTRL_RAW AS t
USING stg_dailytxn_control AS s
ON FALSE
WHEN NOT MATCHED THEN INSERT (
    RECTYPE,
    SUPPLIER_CD,
    FILE_ID,
    EXTRACT_DT,
    REC_CNT,
    DR_TOTAL,
    CR_TOTAL,
    CREATED_TMS
)
VALUES (
    s.RECTYPE,
    s.SUPPLIER_CD,
    s.FILE_ID,
    s.EXTRACT_DT,
    s.REC_CNT,
    s.DR_TOTAL,
    s.CR_TOTAL,
    s.CREATED_TMS
);

-- MultiLoad lifecycle cleanup was replaced with cleanup of Databricks temporary objects.
DROP VIEW stg_dailytxn_detail;

-- MultiLoad lifecycle cleanup was replaced with cleanup of Databricks temporary objects.
DROP VIEW stg_dailytxn_control;

-- MultiLoad lifecycle cleanup was replaced with cleanup of Databricks temporary objects.
DROP TABLE stg_dailytxn_raw;
